# smartdur-medication-alert

원본 분석 코드의 공개용 복사본입니다. 원본 셀 번호는 메타데이터에 기록했습니다. 교육용 문제 설명·개별 기록 출력·계정/개인 경로를 제거했습니다. 데이터는 별도 승인 후 확보해야 합니다. 먼저 프로젝트 README를 읽으세요.

In [ ]:
from pathlib import Path
import os
import platform
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name != "smartdur-medication-alert":
    raise RuntimeError("작업 디렉터리를 projects/smartdur-medication-alert 또는 그 notebooks 폴더로 설정하세요.")
DATA_DIR = Path(os.environ.get("HEALTHCARE_DATA_DIR", str(PROJECT_ROOT / "data")))
ALLOW_TRAINING = False  # 학습 셀 실행을 원할 때만 직접 True로 변경


## 분석 단계 1

원본 코드 셀 1. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import run_analysis as analysis
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
from chart_layout import chart_panels, standard_figure


## 분석 단계 6

원본 코드 셀 6. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
if not ALLOW_TRAINING:
    raise RuntimeError("전체 모델링은 ALLOW_TRAINING=True일 때만 실행합니다.")
model_metrics, selected_metrics = analysis.run()
display(model_metrics.round(3))


## 분석 단계 8

원본 코드 셀 8. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
import json
display(pd.read_csv(analysis.OUT / "label_basis_summary.csv"))
display(pd.read_csv(analysis.OUT / "split_stages.csv"))
display(json.loads((analysis.OUT / "label_audit_summary.json").read_text(encoding="utf-8")))
display(json.loads((analysis.OUT / "reviewed_label_comparison_status.json").read_text(encoding="utf-8")))


## 분석 단계 10

원본 코드 셀 10. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
metrics = pd.read_csv(analysis.OUT / "model_metrics.csv")
chart = metrics.assign(error_plus=metrics.ci_high-metrics.PR_AUC_AP,
                       error_minus=metrics.PR_AUC_AP-metrics.ci_low)
fig = px.bar(chart, x="model", y="PR_AUC_AP", color="model", facet_col="split",
             error_y="error_plus", error_y_minus="error_minus", range_y=[0, 1],
             title="모델별 PR-AUC(AP)와 환자 단위 95% CI")
fig.add_hline(y=0.29, line_dash="dash", annotation_text="이전 X40 참고선 0.29")
for view in chart_panels("model_comparison", fig, analysis.OUT):
    display(view)
display(pd.read_csv(analysis.OUT / "paired_model_differences.csv").round(3))


## 분석 단계 12

원본 코드 셀 12. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
thresholds = pd.read_csv(analysis.OUT / "threshold_metrics.csv")
long = thresholds.melt(id_vars=["split", "model", "threshold"],
                       value_vars=["precision", "recall", "popup_reduction"],
                       var_name="metric", value_name="value")
for view in chart_panels("threshold_tradeoffs", None, analysis.OUT):
    display(view)
display(pd.read_csv(analysis.OUT / "selected_threshold_metrics.csv").round(3))


## 분석 단계 14

원본 코드 셀 14. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
importance = pd.read_csv(analysis.OUT / "shap_top_features.csv")
display(standard_figure(px.bar(importance.head(12).sort_values("mean_abs_shap"), x="mean_abs_shap",
               y="screen_label", orientation="h", title="SHAP 상위 변수: 처방 화면 확인 항목")))
display(importance.head(10))


## 분석 단계 16

원본 코드 셀 16. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
display(pd.read_csv(analysis.OUT / "selection_bias.csv").round(3))
display(pd.read_csv(analysis.OUT / "sensitivity_LR.csv").round(3))
display(pd.read_csv(analysis.OUT / "label_and_time_sensitivity.csv").round(3))
display(pd.read_csv(analysis.OUT / "subgroup_metrics.csv").round(3))


## 분석 단계 18

원본 코드 셀 18. 계산식과 모델 설정은 원본을 유지합니다.

In [ ]:
display(Markdown((analysis.OUT / "분석결과_보고서.md").read_text(encoding="utf-8")))
print("대화형 시각화:", analysis.OUT / "dashboard.html")
